# OFAT hyper-parameter sensitivity — the tuned models, one knob at a time

Anchors on each model's Optuna winner (`<Model>_best.json`), sweeps **one
hyper-parameter at a time** over the range that was searched, and trains every
swept point with **5 seeds**. About **1 200 trainings** in total, so plan on
more than one session — the sweep **resumes**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Steps 2–5 are setup, step 6 is the sweep (the long one), step 7 makes the
figures. `sensitivity/README.md` explains what the curves do and do not show.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > T4 GPU. A sweep on CPU is not realistic.'

## 2. Mount Drive

Point `DRIVE_DIR` at the folder the tuning study wrote — the sweep reads its
`<Model>_best.json` anchors from there and writes its results beside them, so
the tuning and the sensitivity analysis stay together.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ProjectC_tuning'   # <- the tuning results folder
OUT_CSV   = DRIVE_DIR + '/ofat_results.csv'
FIG_DIR   = DRIVE_DIR + '/ofat_figures'

import glob, os
os.makedirs(FIG_DIR, exist_ok=True)
anchors = sorted(os.path.basename(p) for p in glob.glob(DRIVE_DIR + '/*_best.json'))
print(f'{len(anchors)} anchor(s) in {DRIVE_DIR}:')
print('  ' + '\n  '.join(anchors) if anchors else '  none — run tuning/colab_tune.ipynb first')

## 3. Clone the repository

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/gallant-newton-rqkvxz'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Same set the tuning notebook installs — `fast_pytorch_kmeans` for AdaWaveNet,
`reformer-pytorch`/`local-attention`
because `layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 5. The plan, then a smoke test

`--dry_run` prints every configuration the sweep will train — and every
`run.py` command — without training anything. Read the counts before
committing a session to it.

In [ ]:
!python sensitivity/ofat_sensitivity.py --results_dir "$DRIVE_DIR" --out "$OUT_CSV" --dry_run 2>&1 | head -20

In [ ]:
# ~1 minute: build every planned configuration and push one batch through it,
# so a corner the architecture rejects shows up now and not at hour six
!python sensitivity/ofat_sensitivity.py --results_dir "$DRIVE_DIR" --out "$OUT_CSV" \
    --validate 2>&1 | grep -E 'INVALID|Validated|rejected'

In [ ]:
# ~2 minutes: the cheapest model, 2 seeds x 5 epochs, into a scratch file
!python sensitivity/ofat_sensitivity.py --models DLinear --params moving_avg --quick \
    --results_dir "$DRIVE_DIR" --out /content/_smoketest.csv \
    --checkpoint_dir /content/_ckpt 2>&1 | grep -E '^\[OK\]|^\[WARN\]|plan'

## 6. The sweep

One model per call, so progress is saved after each — and every point is
appended to the CSV as it finishes, so **re-running this cell after a
disconnect continues where it stopped**. A point already in the CSV is
skipped; a point that failed is recorded in `<out>.failures.csv` and skipped
too (pass `--retry_failed` to try it again).

Cheapest first, so a short session still produces complete models. Rough cost
on a T4: DLinear/FITS minutes, TSLANet/PatchTST/iTransformer/ModernTCN/
AdaWaveNet/TimeMixer a few hours each, MSGNet the better part of a day.

Checkpoints go to local disk (`/content/_ckpt`), never Drive — they are
rewritten every improving epoch and deleted after each point, so a network
mount would dominate the runtime.

`--itr 3` costs about 40% less and still gives a spread; `--params
learning_rate batch_size lradj` sweeps only the block every model shares.

In [ ]:
MODELS = ['DLinear', 'FITS', 'TSLANet', 'iTransformer', 'PatchTST',
          'ModernTCN', 'AdaWaveNet', 'TimeMixer', 'MSGNet']
ITR    = 5     # seeds per point

import subprocess, time
for model in MODELS:
    print(f'\n{"="*72}\n  {model}\n{"="*72}', flush=True)
    started = time.time()
    subprocess.run(['python', 'sensitivity/ofat_sensitivity.py',
                    '--models', model,
                    '--itr', str(ITR),
                    '--results_dir', DRIVE_DIR,
                    '--out', OUT_CSV,
                    '--checkpoint_dir', '/content/_ckpt'])
    print(f'{model} finished in {(time.time()-started)/60:.1f} min', flush=True)

## 7. Figures and tables

Safe to run on a partial CSV — only the points already recorded are plotted,
so this is also how to look at the sweep while it is still going.

In [ ]:
!python sensitivity/ofat_plots.py --csv "$OUT_CSV" --results_dir "$DRIVE_DIR" \
    --fig_dir "$FIG_DIR" --metrics mse qlike --format png pdf

### The cross-model views

In [ ]:
from IPython.display import Image, display
for name in ('ofat_tornado_mse.png', 'ofat_heatmap_mse.png'):
    path = os.path.join(FIG_DIR, name)
    if os.path.exists(path):
        display(Image(path))

### One panel figure per model

In [ ]:
import glob
for path in sorted(glob.glob(os.path.join(FIG_DIR, 'ofat_*_mse.png'))):
    if 'tornado' in path or 'heatmap' in path:
        continue
    print(os.path.basename(path))
    display(Image(path))

### Which knobs actually moved the metric

`span_pct` is how far the metric travelled across a knob's grid, as a % of the
tuned value; `noise_pct` is 1 sd of the anchor over its seeds. A knob whose
span does not clear the noise has shown **no effect**, which is not the same
statement as a small one. `gain_pct` is how much better than the anchor the
knob's best grid value was — 0 means the tuned value won its own sweep.

In [ ]:
import pandas as pd
summary = pd.read_csv(os.path.join(FIG_DIR, 'ofat_summary_mse.csv'))
top = (summary.sort_values('span_pct', ascending=False)
              [['model', 'param', 'anchor_value', 'best_value',
                'span_pct', 'gain_pct', 'noise_pct', 'above_noise']])
display(top.head(25).round(3))

In [ ]:
# knobs where the search left something on the table: a better value exists,
# by more than seed noise
left = summary[(summary['gain_pct'] > summary['noise_pct'])]
display(left.sort_values('gain_pct', ascending=False)
            [['model', 'param', 'anchor_value', 'best_value',
              'gain_pct', 'noise_pct']].round(3))

## Notes

* **What a curve means** — every other knob is held at the tuned optimum, so a
  panel measures *local* sensitivity. Interactions are invisible to OFAT by
  construction; that is the price of dozens of runs instead of thousands.
* **Read against seed noise** — the grey band on every panel is the anchor
  ±1 sd over its seeds. A curve inside it has not been shown to matter.
* **The anchor is one run** — trained once per model and reused as the centre
  of every panel, not retrained per knob.
* **One machine per sweep** — a CSV half-trained on CPU and half on GPU
  compares configurations across hardware, not against each other.
* **Adding a model** — drop its `<Model>_best.json` into `DRIVE_DIR` and add
  its name to `MODELS`. Grids for all ten, including TimesNet,
  are already in `sensitivity/ofat_grids.py`.